# Pipeline completo — Tempo Útil pela BOLA (+ jogadores + áudio de referência)
Liga tudo: detetor de bola treinado (`best.pt`) + jogadores + pancadas por áudio (só referência) → `rallies_bola` → tempo útil + corte.
Compara com o ground-truth do corte manual (**132s**).

## 1. Setup

In [ ]:
!pip install -q ultralytics
from google.colab import drive; drive.mount('/content/drive')
import os, numpy as np
# clonar o repo do GitHub — traz sempre a ultima versao dos modulos (rallies_bola.py, audio_pancadas.py)
!rm -rf /content/Modelo-Padel-VRO && git clone -q https://github.com/Vasco-Rocha/Modelo-Padel-VRO.git /content/Modelo-Padel-VRO
REPO='/content/Modelo-Padel-VRO'
import sys; sys.path.insert(0, REPO)
VIDEO='/content/drive/MyDrive/PadelPro_Vision/videos/Analise Padel Modelo - Parada 4 min.mp4'
BALL_PT='/content/drive/MyDrive/PadelPro_Vision/weights/ball_yolo/best.pt'
print('video?', os.path.exists(VIDEO), '| best.pt?', os.path.exists(BALL_PT), '| modulos?', os.path.exists(REPO+'/padelpro/modules/rallies_bola.py'))

## 2. Detetar BOLA por frame (com filtro de hotspots estáticos)

In [ ]:
import cv2
from collections import Counter
from ultralytics import YOLO
mbola=YOLO(BALL_PT)
CONF=0.4; G=40
cap=cv2.VideoCapture(VIDEO); fps=cap.get(cv2.CAP_PROP_FPS); N=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
centers=[]
while True:
    ok,fr=cap.read()
    if not ok: break
    r=mbola.predict(fr, conf=CONF, imgsz=960, verbose=False)[0]
    cs=[((int(b.xyxy[0][0])+int(b.xyxy[0][2]))//2,(int(b.xyxy[0][1])+int(b.xyxy[0][3]))//2) for b in r.boxes]
    centers.append(cs)
cap.release()
# filtro hotspots estáticos (saco/risca fixa): célula com deteção em >20% dos frames = fixo
cellf=lambda c:(c[0]//G,c[1]//G); cnt=Counter()
for cs in centers:
    for c in set(cellf(p) for p in cs): cnt[c]+=1
static={k for k,v in cnt.items() if v>0.20*len(centers)}
# ball_xy = 1 ponto por frame (o 1º detetado que NÃO seja estático), senão None
ball_xy=[]
for cs in centers:
    p=next((p for p in cs if cellf(p) not in static), None)
    ball_xy.append(p)
print('hotspots removidos:',len(static),'| frames com bola:',sum(p is not None for p in ball_xy),'/',N)

## 3. Detetar JOGADORES por frame (YOLOv8 pessoas)

In [ ]:
mppl=YOLO('yolov8n.pt')   # classe 0 = pessoa
cap=cv2.VideoCapture(VIDEO); player_boxes=[]
while True:
    ok,fr=cap.read()
    if not ok: break
    r=mppl.predict(fr, conf=0.4, classes=[0], imgsz=640, verbose=False)[0]
    boxes=[(float(b.xyxy[0][0]),float(b.xyxy[0][1]),float(b.xyxy[0][2]),float(b.xyxy[0][3])) for b in r.boxes]
    boxes=sorted(boxes,key=lambda b:-(b[2]-b[0])*(b[3]-b[1]))[:4]   # 4 maiores = jogadores
    player_boxes.append(boxes)
cap.release()
print('frames com jogadores:',sum(len(b)>0 for b in player_boxes))

## 4. Pancadas por ÁUDIO (só REFERÊNCIA — pode ter atraso/ruído de outros campos)

In [ ]:
import subprocess
subprocess.run(['ffmpeg','-y','-i',VIDEO,'-ac','1','-ar','16000','-vn','/content/aud.wav'],capture_output=True)
from padelpro.modules.audio_pancadas import detetar_pancadas_audio
audio_hits,dur=detetar_pancadas_audio('/content/aud.wav')
print('candidatos a pancada (áudio):',len(audio_hits),'| densidade',round(60*len(audio_hits)/dur,1),'/min')

## 5. Segmentar rallies (todas as regras da bola) + comparar com 132s

In [ ]:
from padelpro.modules.rallies_bola import segmentar_rallies_bola
res=segmentar_rallies_bola(ball_xy, player_boxes, fps, audio_hits_s=audio_hits)
print(f"{res['n_rallies']} rallies | tempo útil {res['tempo_util_s']}s / {round(N/fps)}s ({round(100*res['tempo_util_s']/(N/fps))}%)")
print('GROUND-TRUTH (corte manual): 132s (45%)  <- comparar')
for r in res['rallies'][:30]: print(r)
print('trocas_de_campo:',res['trocas_de_campo'])

## 6. Cortar o vídeo (clips com margem R8 de +2s) e guardar na Drive

In [ ]:
os.makedirs('/content/clips',exist_ok=True); parts=[]
for j,(a,b) in enumerate(res['clips_video']):
    o=f'/content/clips/r{j:02d}.mp4'
    subprocess.run(['ffmpeg','-y','-ss',f'{a/fps:.2f}','-to',f'{b/fps:.2f}','-i',VIDEO,'-c:v','libx264','-crf','23','-preset','veryfast','-an',o],capture_output=True)
    parts.append(os.path.abspath(o))
open('/content/list.txt','w').write('\n'.join(f"file '{p}'" for p in parts))
OUT='/content/drive/MyDrive/PadelPro_Vision/jogos/parada4_teste/TempoUtil_pipeline.mp4'
os.makedirs(os.path.dirname(OUT),exist_ok=True)
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/content/list.txt','-c','copy',OUT],capture_output=True)
print('condensado na Drive:',os.path.exists(OUT))